In [36]:
!pip install openai pandas scikit-learn -q

In [37]:
import os
import pandas as pd
from openai import OpenAI

In [38]:
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API')
except Exception:
    OPENROUTER_API_KEY = os.getenv('OPENROUTER_API', 'sk-or-YOUR-KEY-HERE')

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "arcee-ai/trinity-large-preview:free"

print(f"Model  : {MODEL}")
print(f"API key: {'SET' if OPENROUTER_API_KEY and 'YOUR-KEY' not in OPENROUTER_API_KEY else 'NOT SET'}")

Model  : arcee-ai/trinity-large-preview:free
API key: SET


In [39]:
df = pd.read_csv("Telco_Customer_Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [40]:
df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

df["Churn"] = df["Churn"].map({"Yes":1,"No":0})

/tmp/ipykernel_327/3336407893.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [43]:
from sklearn.preprocessing import LabelEncoder

df_ml = df.copy()

le = LabelEncoder()

for col in df_ml.columns:
    if df_ml[col].dtype == "object":
        df_ml[col] = le.fit_transform(df_ml[col])

In [44]:
from sklearn.model_selection import train_test_split

X = df_ml.drop("Churn",axis=1)
y = df_ml["Churn"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [45]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=2000)

lr_model.fit(X_train,y_train)

LogisticRegression(max_iter=2000)

In [46]:
from sklearn.metrics import accuracy_score

y_pred = lr_model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.8161816891412349


In [47]:
importance_lr = pd.Series(
    abs(lr_model.coef_[0]),
    index=X.columns
).sort_values(ascending=False)

print(importance_lr.head(10))


PhoneService        1.023004
Contract            0.724697
PaperlessBilling    0.356779
OnlineSecurity      0.292292
TechSupport         0.256222
InternetService     0.228458
Dependents          0.185089
SeniorCitizen       0.178276
OnlineBackup        0.155914
DeviceProtection    0.082445
dtype: float64


In [48]:
def build_context(df):

    total = len(df)

    churn_rate = df["Churn"].mean()*100

    ctx = f"""Telco Churn Dataset — {total} customers, {len(df.columns)} columns
Columns: {', '.join(df.columns)}

Churn rate: {churn_rate:.1f}%

Model accuracy: {accuracy:.2f}

Top churn drivers:
{importance_lr.head(5).to_string()}

Churn by Contract:
{df.groupby('Contract')['Churn'].mean().mul(100).round(1).astype(str)+'%'}

Avg Monthly Charges:
Churned: ${df[df['Churn']==1]['MonthlyCharges'].mean():.2f}
Retained: ${df[df['Churn']==0]['MonthlyCharges'].mean():.2f}

Avg Tenure:
Churned: {df[df['Churn']==1]['tenure'].mean():.1f}
Retained: {df[df['Churn']==0]['tenure'].mean():.1f}

Churn by Internet Service:
{df.groupby('InternetService')['Churn'].mean().mul(100).round(1).astype(str)+'%'}

Churn by Payment Method:
{df.groupby('PaymentMethod')['Churn'].mean().mul(100).round(1).astype(str)+'%'}
"""

    return ctx

In [49]:
CONTEXT = build_context(df)

print(CONTEXT)

Telco Churn Dataset — 7043 customers, 20 columns
Columns: gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges, Churn

Churn rate: 26.5%

Model accuracy: 0.82

Top churn drivers:
PhoneService        1.023004
Contract            0.724697
PaperlessBilling    0.356779
OnlineSecurity      0.292292
TechSupport         0.256222

Churn by Contract:
Contract
Month-to-month    42.7%
One year          11.3%
Two year           2.8%
Name: Churn, dtype: object

Avg Monthly Charges:
Churned: $74.44
Retained: $61.27

Avg Tenure:
Churned: 18.0
Retained: 37.6

Churn by Internet Service:
InternetService
DSL            19.0%
Fiber optic    41.9%
No              7.4%
Name: Churn, dtype: object

Churn by Payment Method:
PaymentMethod
Bank transfer (automatic)    16.7%
Credit card (automatic)      15.2%

In [50]:
def ask(question):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a data analyst. Answer using only the dataset stats provided."},
            {"role": "user",   "content": f"Dataset:\n{CONTEXT}\n\nQuestion: {question}"}
        ],
        max_tokens=300,
    )

    print(f"Q: {question}")
    print("-" * 50)
    print(response.choices[0].message.content.strip())
    print()

In [52]:
ask("Describe the dataset")




Q: Describe the dataset
--------------------------------------------------
The Telco Churn Dataset contains 7,043 customer records with 20 columns covering demographic information (gender, SeniorCitizen, Partner, Dependents), service details (tenure, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies), contract terms (Contract, PaperlessBilling), payment information (PaymentMethod), and billing data (MonthlyCharges, TotalCharges). The dataset shows a churn rate of 26.5%, with the model achieving 82% accuracy in predicting churn. Key churn drivers include PhoneService, Contract type, PaperlessBilling, OnlineSecurity, and TechSupport. The data reveals significant patterns in customer behavior, such as higher churn rates for month-to-month contracts (42.7%), fiber optic internet users (41.9%), and electronic check payment method users (45.3%). Churned customers typically have higher monthly charges ($74.4

In [53]:
ask("What are the high-risk customer characteristics?")


Q: What are the high-risk customer characteristics?
--------------------------------------------------
Based on the dataset statistics, high-risk customer characteristics include:

1. Month-to-month contract holders (42.7% churn rate)
2. Fiber optic internet service users (41.9% churn rate)
3. Electronic check payment method users (45.3% churn rate)
4. No phone service (implied by low coefficient of 1.023004)
5. No online security (coefficient of 0.292292)
6. No tech support (coefficient of 0.256222)
7. Paperless billing (coefficient of 0.356779)
8. Lower tenure (average of 18 months for churned customers vs 37.6 for retained)
9. Higher monthly charges ($74.44 for churned vs $61.27 for retained)
10. No partner or dependents

These characteristics indicate customers who are more likely to churn, with month-to-month contract holders, fiber optic users, and electronic check payers being the highest-risk groups.



In [54]:
ask("Which payment method has the highest churn?")

Q: Which payment method has the highest churn?
--------------------------------------------------
Electronic check has the highest churn rate at 45.3%.

